# Trabalho 1: Panorama a partir de Imagens Fora de Ordem

Versão em notebook do `panorama.py`. Executa as mesmas etapas, com os mesmos módulos, e justifica as escolhas de cada uma.

| Passo | Etapa do enunciado | Pasta |
|---|---|---|
| 1. Detecção de características | Etapa 2 | `step1_key_points_detection/` |
| 2. Emparelhamento | Etapa 3 | `step2_feature_matching/` |
| 3. Ordenação automática | Etapa 4 | `step3_image_sorting/` |
| 4. Homografia e alinhamento | Etapa 5 | `step4_homography_connection/` |
| 5. Composição e remoção de fantasmas | Etapa 6 | `step5_panorama_composition/` |

**Como usar:** defina `GRUPO` na célula de configuração e execute todas as células. Funciona para qualquer pasta de `input/`. As saídas de cada passo vão para `output/<passo>/<grupo>/`, e o panorama, para `output/final/<grupo>/panorama.jpg`.

## Configuração

In [ ]:
from pathlib import Path

from IPython.display import Image, display

from common import OUTPUT_DIR, load_image_set, print_table, save_image, set_seed, stage_output_dir
from panorama import print_capture_metadata
from step1_key_points_detection import detector as passo1
from step2_feature_matching import matcher as passo2
from step3_image_sorting import sorter as passo3
from step4_homography_connection import alignment as passo4
from step5_panorama_composition import compositor as passo5
from step5_panorama_composition import evaluation as avaliacao5

Parâmetros do pipeline. Os valores padrão são os escolhidos nos estudos de cada passo (justificativas abaixo, em cada seção).

- `GRUPO`: nome de uma pasta de `input/`.
- `PAR_COMPARACAO`, `REGIAO_FANTASMA` e `REGIAO_COSTURA` dependem do grupo. Com `None`, são escolhidos automaticamente.

In [ ]:
GRUPO = "paisagem"                # pasta de input/ a processar

LADO_MAXIMO = 1024                # px; as fotos são reduzidas (velocidade, sem perda para o alinhamento)
SEMENTE_EMBARALHAMENTO = 0        # embaralha a ordem de entrada (requisito 4.1); None = ordem alfabética
MAX_KEYPOINTS = 3000              # keypoints mantidos por imagem (os de maior resposta)
DETECTOR = "SIFT"                 # SIFT, ORB ou AKAZE
LIMIAR_RATIO = passo2.RATIO       # ratio test de Lowe (0.75)
MIN_INLIERS = passo3.MIN_INLIERS  # inliers mínimos para duas imagens se sobreporem
PROJECAO = "cylindrical"          # cylindrical ou planar
ALINHAMENTO = "pairwise"          # pairwise (par a par) ou bundle (bundle adjustment)
REMOCAO_FANTASMAS = "seam"        # seam (costura ótima) ou none
BLENDING = "feather"              # feather, multiband ou none

PAR_COMPARACAO = None             # ex.: ("20260926_3", "20260926_4"); par exibido com os 3 detectores
REGIAO_FANTASMA = None            # (x, y, w, h) no panorama final; região ampliada sem/com remoção de fantasmas
REGIAO_COSTURA = None             # (x, y, w, h) no panorama final; região ampliada nos métodos de composição

PASTA = Path("input") / GRUPO


def mostrar(passo: str, arquivo: str, largura: int = 1000) -> None:
    """Exibe uma figura salva em output/<passo>/<grupo>/."""
    display(Image(filename=str(OUTPUT_DIR / passo / GRUPO / arquivo), width=largura))

## Entrada (Etapa 1 e requisito 4.1)

As imagens são carregadas em **ordem embaralhada**. O pipeline usa apenas os pixels: nomes de arquivo servem só como rótulo e nenhum metadado EXIF de tempo é lido. O arquivo `metadados.json` do grupo (item 1.4), se existir, é apenas exibido.

In [ ]:
set_seed()
imagens = load_image_set(PASTA, LADO_MAXIMO, SEMENTE_EMBARALHAMENTO)
print_capture_metadata(PASTA, GRUPO)

## Passo 1: Detecção e Extração de Características (Etapa 2)

Os três detectores são executados em todas as imagens e comparados em quantidade de keypoints, tempo, cobertura e uniformidade espacial.

**Escolha: SIFT.**
- Invariante a escala e rotação, com descritor de 128 valores reais, mais discriminativo que os binários do ORB e do AKAZE.
- Distribui os keypoints de forma mais uniforme pela imagem, o que estabiliza a homografia.
- O custo maior (~40 ms por imagem) é irrelevante para poucas imagens.
- A qualidade do emparelhamento, medida no passo 2, confirma a escolha.

In [ ]:
caracteristicas = passo1.compare_detectors(imagens, stage_output_dir(passo1.STAGE, GRUPO), MAX_KEYPOINTS)

In [ ]:
mostrar(passo1.STAGE, "comparacao_detectores.png")
mostrar(passo1.STAGE, f"detectores_{imagens.names[0]}.png")

## Passo 2: Emparelhamento de Características (Etapa 3)

**Brute Force em vez de FLANN.**
- É exato: encontra os dois vizinhos mais próximos reais, exigidos pelo ratio test.
- É determinístico e, com ~3000 descritores por imagem, mais rápido que construir um índice FLANN para cada par.
- Usa norma L2 para SIFT e Hamming para ORB e AKAZE.

**Ratio test de Lowe com limiar 0.75:** um match é aceito se a distância ao vizinho mais próximo for menor que 0.75 vez a distância ao segundo. Lowe (2004) sugere 0.8, mas a curva do ratio test (SIFT, média sobre os pares com sobreposição) mostra que 0.75 é melhor nos dois conjuntos:

| Conjunto | Precisão com 0.75 | Precisão com 0.8 | Matches aceitos (0.75 → 0.8) |
|---|---|---|---|
| paisagem | 95.9% | 93.3% | 700 → 727 (+4%) |
| igreja | 74.4% | 65.2% | 212 → 253 (+19%) |

Passar para 0.8 ganha poucos matches, e a maior parte deles é incorreta. Com 0.75 sobram matches mais que suficientes para estimar as homografias, e a cena difícil (igreja, noturna e com paralaxe) é a mais beneficiada.

A precisão (inliers / aceitos) usa um RANSAC **apenas como régua de avaliação**, porque não há gabarito dos matches corretos.

O estudo precisa saber quais pares se sobrepõem, então a ordenação do passo 3 é calculada antes.

In [ ]:
ordenacao = passo3.ImageSorter(passo2.FeatureMatcher(LIMIAR_RATIO), MIN_INLIERS).sort(
    imagens, caracteristicas[DETECTOR])
passo2.compare_matching(imagens, stage_output_dir(passo2.STAGE, GRUPO), caracteristicas, ordenacao.overlap_edges,
                        ordenacao.order, LIMIAR_RATIO, DETECTOR, PAR_COMPARACAO);

In [ ]:
mostrar(passo2.STAGE, "comparacao_emparelhamento.png")
mostrar(passo2.STAGE, "curva_ratio_test.png", 800)
a, b = ordenacao.order[0], ordenacao.order[1]
mostrar(passo2.STAGE, f"matches_{'_'.join(sorted((imagens.names[a], imagens.names[b])))}.jpg")

## Passo 3: Ordenação Automática e Rejeição de Intrusas (Etapa 4)

1. **Matriz de conectividade:** número de inliers RANSAC para cada par de imagens.
2. **Sobreposição:** o par se sobrepõe se `inliers ≥ max(20, 8 + 0.3 × matches)`, critério de Brown & Lowe (2007).
3. **Intrusas:** imagens fora do maior componente conexo do grafo são rejeitadas.
4. **Sequência:** caminho mais longo da árvore geradora máxima, orientado da esquerda para a direita.
5. **Referência:** a imagem central da sequência, que minimiza a distorção.

In [ ]:
passo3.save_sorting_outputs(imagens, ordenacao, stage_output_dir(passo3.STAGE, GRUPO))

In [ ]:
mostrar(passo3.STAGE, "matriz_conectividade.png", 700)
mostrar(passo3.STAGE, "grafo_vizinhanca.png")
mostrar(passo3.STAGE, "sequencia_inferida.jpg")
for k in ordenacao.rejected:
    mostrar(passo3.STAGE, f"intrusa_{imagens.names[k]}.jpg")

## Passo 4: Estimação de Homografia e Alinhamento (Etapa 5)

- **Homografia por par:** `findHomography` com RANSAC (limiar de 3 px) sobre os matches aceitos.
- **Projeção cilíndrica:** conjuntos com campo de visão amplo (> ~120°) tornam a projeção planar inviável, porque o canvas cresce sem limite. A distância focal é estimada automaticamente, escolhendo o valor que torna `K⁻¹·H·K` mais próxima de uma rotação.
- **Encadeamento par a par:** cada imagem é ligada à vizinha na árvore, a partir da referência.
- **Bundle adjustment (extra X1):** refina todas as transformações ao mesmo tempo, usando todas as sobreposições. Nos pares vizinhos, o erro sobe ligeiramente, porque o par a par é ótimo neles por construção. Em todas as sobreposições, a deriva acumulada diminui.

In [ ]:
alinhamento = passo4.align(imagens.images, ordenacao.pairs, ordenacao.tree_edges, ordenacao.overlap_edges,
                           ordenacao.reference, ALINHAMENTO, PROJECAO, compare_modes=True)
passo4.save_alignment_outputs(imagens.names, alinhamento, ordenacao.order, ordenacao.overlap_edges, GRUPO,
                              stage_output_dir(passo4.STAGE, GRUPO))

In [ ]:
mostrar(passo4.STAGE, "metricas_homografias.png")
mostrar(passo4.STAGE, "alinhamento_progressivo.jpg")
mostrar(passo4.STAGE, "contornos_imagens.jpg")

## Passo 5: Composição, Blending e Remoção de Fantasmas (Etapa 6)

- **Costura ótima (programação dinâmica):** cada imagem entra no mosaico pelo caminho de menor diferença de cor na sobreposição. Cada pixel vem de uma única imagem, o que elimina os fantasmas de objetos em movimento.
- **Feathering:** suaviza apenas uma faixa estreita em torno da costura, escondendo o corte sem reintroduzir fantasmas.
- **Métodos comparados:** média simples, feathering, costura, costura + feathering e costura + multibanda. A costura + feathering é a única que tem, ao mesmo tempo, fantasmas e descontinuidade na costura baixos.
- **Recorte final:** o panorama é cortado no maior retângulo sem bordas pretas.

In [ ]:
pasta5 = stage_output_dir(avaliacao5.STAGE, GRUPO)
compositor = passo5.PanoramaCompositor(REMOCAO_FANTASMAS, BLENDING)
composicao = compositor.compose(alinhamento.aligned)
avaliacao5.save_composition_outputs(imagens.names, alinhamento, compositor.name, composicao, GRUPO, pasta5,
                                    REGIAO_FANTASMA)
avaliacao5.compare_compositions(alinhamento, REGIAO_COSTURA, GRUPO, pasta5)

In [ ]:
mostrar(avaliacao5.STAGE, "comparacao_deghosting.jpg")
mostrar(avaliacao5.STAGE, "comparacao_metodos.jpg")
mostrar(avaliacao5.STAGE, "metricas_metodos_composicao.png")
mostrar(avaliacao5.STAGE, "costuras.jpg")

## Panorama Final

In [ ]:
caminho = OUTPUT_DIR / "final" / GRUPO / "panorama.jpg"
save_image(caminho, composicao.panorama)
h, w = composicao.panorama.shape[:2]
print_table([
    {"item": "Imagens na entrada", "valor": len(imagens)},
    {"item": "Sequência inferida", "valor": " -> ".join(imagens.names[k] for k in ordenacao.order)},
    {"item": "Intrusas rejeitadas", "valor": ", ".join(imagens.names[k] for k in ordenacao.rejected) or "nenhuma"},
    {"item": "Imagem de referência", "valor": imagens.names[ordenacao.reference]},
    {"item": "Panorama", "valor": f"{w}x{h} px, salvo em {caminho}"},
], f"Resumo ({GRUPO})")
display(Image(filename=str(caminho), width=1200))